In [ ]:
import os

os.environ["AWS_ACCESS_KEY_ID"] = ""
os.environ["AWS_SECRET_ACCESS_KEY"] = ""
os.environ["AWS_DEFAULT_REGION"] = "ru-central1"
os.environ["AWS_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
print("✓ Переменные окружения установлены")

✓ Переменные окружения установлены


## Обучаем (несколько эпох) ResNet18 на новом датасете

In [ ]:
!pip install -q lightning
!pip install litdata

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 58.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 205.4/205.4 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 74.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 67.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 10.5 MB/s eta 0:00:00


In [ ]:
!git clone https://github.com/xan-d-or/sneakers-hse.git
!cd sneakers-hse
!git checkout yolo

Cloning into 'sneakers-hse'...
remote: Enumerating objects: 542, done.
remote: Counting objects: 100% (139/139), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 542 (delta 70), reused 87 (delta 49), pack-reused 403 (from 2)
Receiving objects: 100% (542/542), 105.16 MiB | 29.81 MiB/s, done.
Resolving deltas: 100% (229/229), done.
fatal: not a git repository (or any of the parent directories): .git


In [ ]:
!cd sneakers-hse && git checkout yolo

Already on 'yolo'
Your branch is up to date with 'origin/yolo'.


In [ ]:
!cd sneakers-hse && git pull

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 450 bytes | 450.00 KiB/s, done.
From https://github.com/xan-d-or/sneakers-hse
   a1d974c..12b870a  yolo       -> origin/yolo
Updating a1d974c..12b870a
Fast-forward
 src/data/__init__.py | 4 ++--
 1 file changed, 2 insertions(+), 2 deletions(-)


In [ ]:
!ls -la .

total 20
drwxr-xr-x 1 root root 4096 Apr  6 19:16 .
drwxr-xr-x 1 root root 4096 Apr  6 18:05 ..
drwxr-xr-x 4 root root 4096 Mar 30 13:34 .config
drwxr-xr-x 1 root root 4096 Mar 30 13:34 sample_data
drwxr-xr-x 9 root root 4096 Apr  6 19:16 sneakers-hse


In [ ]:
import sys
sys.path.append('./sneakers-hse')
sys.path.append('/content/sneakers-hse')
sys.path.append('/content/sneakers-hse/src')

from pathlib import Path
import os
from glob import glob

import pandas as pd
import numpy as np

from PIL import Image

from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import pytorch_lightning as pl
from torch.utils.data import DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

from src.model.baseline_cnn import LitCNN
from src.model.resnet_18 import LitResNet18
from src.model.dataset import ImageDataset
from src.model.streaming_dataset import StreamingImageDataset
from src.model.classifier import LitClassifier

from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

from litdata import optimize
import fsspec

from tqdm import tqdm
tqdm.pandas()

# %load_ext autoreload
# %autoreload 2

In [ ]:
# from src.data.dataset import LitDataImageDataset
# from src.data.s3_client import YandexS3Client
# from src.data.label2idx import build_label2idx_s3

from src.data import build_label2idx_s3, YandexS3Client, LitDataImageDataset

In [ ]:
transforms = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),  # Только для train
    A.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
    ToTensorV2(),
], is_check_shapes=False)


In [ ]:
client = YandexS3Client(
    access_key=os.getenv("AWS_ACCESS_KEY_ID"),
    secret_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    bucket_name=os.getenv("AWS_BUCKET_NAME"),
    endpoint=os.getenv("AWS_ENDPOINT_URL"),
    region=os.getenv("AWS_DEFAULT_REGION"),
)

In [ ]:
label2idx = None

with open("/content/label2idx.pkl", "rb") as f:
    label2idx = pd.read_pickle(f)

In [ ]:
s3_path = "s3://sneakers-hse-images-test/yolo_preprocessed/train"

dataset = LitDataImageDataset(
    input_dir=s3_path,
    label2idx=label2idx,
    transform=transforms,
    # cache_dir=CACHE_DIR,
)

In [ ]:
import boto3
import io
from io import BytesIO

In [ ]:
s3 = boto3.client('s3')
obj = s3.get_object(Bucket='sneakers-hse-images-test', Key='yolo_preprocessed/train_images.csv')

# Use io.BytesIO to convert the streaming body to a file-like object
train_df_pre = pd.read_csv(io.BytesIO(obj['Body'].read()))


In [ ]:
obj = s3.get_object(Bucket='sneakers-hse-images-test', Key='yolo_preprocessed/test_images.csv')

# Use io.BytesIO to convert the streaming body to a file-like object
test_df = pd.read_csv(io.BytesIO(obj['Body'].read()))

In [ ]:

display(train_df_pre.head(), train_df_pre.shape)

display(test_df.head(), test_df.shape)

,Unnamed: 0,path,sneaker_class,corrupted_flg
0,0,Vans Кеды Upland/clothing_0_264.jpeg,Vans Кеды Upland,0
1,1,Vans Кеды Upland/clothing_0_57.jpeg,Vans Кеды Upland,0
2,2,Vans Кеды Upland/orig_45.jpeg,Vans Кеды Upland,0
3,3,Vans Кеды Upland/clothing_0_0.jpeg,Vans Кеды Upland,0
4,4,Vans Кеды Upland/clothing_0_233.jpeg,Vans Кеды Upland,0


(10965, 4)

,Unnamed: 0,path,sneaker_class,corrupted_flg
0,14,Vans Кеды Upland/clothing_0_168_real.jpeg,Vans Кеды Upland,0
1,32,Vans Кеды Upland/clothing_0_215_real.jpeg,Vans Кеды Upland,0
2,44,Vans Кеды Upland/orig_216_real.jpeg,Vans Кеды Upland,0
3,80,Vans Кеды Upland/shoe_3_100_real.jpeg,Vans Кеды Upland,0
4,87,Vans Кеды Upland/clothing_0_277_real.jpeg,Vans Кеды Upland,0


(300, 4)

In [ ]:
train_df, val_df = train_test_split(
    train_df_pre,
    test_size=0.2,
    stratify=train_df_pre["sneaker_class"],
    random_state=42
)

display(train_df.head(), train_df.shape)
display(val_df.head(), val_df.shape)
display(test_df.head(), test_df.shape)

,Unnamed: 0,path,sneaker_class,corrupted_flg
2195,2288,Nike Кеды Dunk Low Retro/clothing_0_103.jpeg,Nike Кеды Dunk Low Retro,0
10557,10855,PUMA Кроссовки Puma Morphic/orig_111.jpeg,PUMA Кроссовки Puma Morphic,0
7299,7477,Kari Кроссовки/clothing_0_190.jpeg,Kari Кроссовки,0
4103,4230,Reebok Кроссовки CLASSIC LEATHER/clothing_0_43...,Reebok Кроссовки CLASSIC LEATHER,0
2097,2188,Nike Кеды Dunk Low Retro/clothing_0_86.jpeg,Nike Кеды Dunk Low Retro,0


(8772, 4)

,Unnamed: 0,path,sneaker_class,corrupted_flg
7461,7639,Vans Кеды Knu Skool/clothing_0_98.jpeg,Vans Кеды Knu Skool,0
2486,2591,Reebok Кроссовки CLASSIC NYLON/orig_291.jpeg,Reebok Кроссовки CLASSIC NYLON,0
4889,5027,Nike Кроссовки AIR MAX 90/clothing_0_12.jpeg,Nike Кроссовки AIR MAX 90,0
2655,2762,Under Armour Кроссовки UA CHARGED SPEED SWIFT/...,Under Armour Кроссовки UA CHARGED SPEED SWIFT,0
231,240,Vans Кеды Upland/clothing_0_62.jpeg,Vans Кеды Upland,0


(2193, 4)

,Unnamed: 0,path,sneaker_class,corrupted_flg
0,14,Vans Кеды Upland/clothing_0_168_real.jpeg,Vans Кеды Upland,0
1,32,Vans Кеды Upland/clothing_0_215_real.jpeg,Vans Кеды Upland,0
2,44,Vans Кеды Upland/orig_216_real.jpeg,Vans Кеды Upland,0
3,80,Vans Кеды Upland/shoe_3_100_real.jpeg,Vans Кеды Upland,0
4,87,Vans Кеды Upland/clothing_0_277_real.jpeg,Vans Кеды Upland,0


(300, 4)

In [ ]:
train_paths = train_df["path"].tolist()
val_paths   = val_df["path"].tolist()
test_paths  = test_df["path"].tolist()

train_labels = train_df["sneaker_class"].tolist()
val_labels   = val_df["sneaker_class"].tolist()
test_labels  = test_df["sneaker_class"].tolist()

In [ ]:
s3_prefix = "s3://sneakers-hse-images-test/yolo_preprocessed/search-dataset-images"


train_paths = [s3_prefix + '/' + path for path in train_paths]
val_paths = [s3_prefix + '/' + path for path in val_paths]
test_paths = [s3_prefix + '/' + path for path in test_paths]

In [ ]:
train_input = [{ 'path': path, 'label': label } for path, label in zip(train_paths, train_labels)]
val_input = [{ 'path': path, 'label': label } for path, label in zip(val_paths, val_labels)]
test_input = [{ 'path': path, 'label': label } for path, label in zip(test_paths, test_labels)]

In [ ]:
from litdata import optimize
import PIL

In [ ]:
s3_client = boto3.client(
    service_name='s3',
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    endpoint_url=os.getenv("AWS_ENDPOINT_URL"),
    region_name=os.getenv("AWS_DEFAULT_REGION"),
)

In [ ]:
bucket_name = "sneakers-hse-images-test"
def encode(sample):
    response = s3_client.get_object(Bucket=bucket_name, Key=sample["path"].replace(f"s3://{bucket_name}/", ""))
    image_data = response['Body'].read()
    return {
        "image": PIL.Image.open(BytesIO(image_data)),
        "label": sample["label"],
    }

In [ ]:
encode(train_input[0])

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=4288x1731>,
 'label': 'Nike Кеды Dunk Low Retro'}

In [ ]:
# Optimize train data
optimize(
    fn=encode,
    inputs=train_input,
    output_dir="s3://sneakers-hse-images-test/yolo_preprocessed/train",  # Or S3: "s3://bucket/optimized/train"
    chunk_bytes="64MB",  # Adjust based on your needs
    num_workers=8,  # Parallel workers for speed
    mode='overwrite'
)

# Optimize val data
optimize(
    fn=encode,
    inputs=val_input,
    output_dir="s3://sneakers-hse-images-test/yolo_preprocessed/val",
    chunk_bytes="64MB",
    num_workers=8,
)

# Optimize test data
optimize(
    fn=encode,
    inputs=test_input,
    output_dir="s3://sneakers-hse-images-test/yolo_preprocessed/test",
    chunk_bytes="64MB",
    num_workers=8,
)

Create an account on https://lightning.ai/ to optimize your data faster using multiple nodes and large machines.
Setting multiprocessing start_method to fork. Tip: Libraries relying on lock can hang with `fork`. To use `spawn` in notebooks, move your code to files and import it within the notebook.
Storing the files under s3://sneakers-hse-images-test/yolo_preprocessed/val
Setup started with fast_dev_run=False.
Setup finished in 0.005 seconds. Found 2193 items to process.
Starting 8 workers with 2193 items. The progress bar is only updated when a worker finishes.
Workers are ready ! Starting data processing...


Progress:   0%|          | 0/2193 [00:00<?, ?it/s]

Rank 2 inferred the following `['jpeg', 'str']` data format.
Rank 3 inferred the following `['jpeg', 'str']` data format.
Rank 1 inferred the following `['jpeg', 'str']` data format.
Rank 0 inferred the following `['jpeg', 'str']` data format.
Rank 5 inferred the following `['jpeg', 'str']` data format.
Rank 6 inferred the following `['jpeg', 'str']` data format.
Rank 4 inferred the following `['jpeg', 'str']` data format.
Rank 7 inferred the following `['jpeg', 'str']` data format.
Worker 5 is terminating.
Worker 1 is terminating.
Worker 4 is terminating.
Worker 5 is done.
Worker 0 is terminating.
Worker 6 is terminating.
Worker 3 is terminating.
Worker 2 is terminating.
Worker 1 is done.
Worker 4 is done.
Worker 0 is done.
Worker 7 is terminating.
Worker 2 is done.
Worker 6 is done.
Worker 3 is done.
Worker 7 is done.
Workers are finished.
Finished data processing!
Create an account on https://lightning.ai/ to optimize your data faster using multiple nodes and large machines.
Setting

Progress:   0%|          | 0/300 [00:00<?, ?it/s]

Rank 0 inferred the following `['jpeg', 'str']` data format.
Rank 3 inferred the following `['jpeg', 'str']` data format.
Rank 2 inferred the following `['jpeg', 'str']` data format.
Rank 6 inferred the following `['jpeg', 'str']` data format.
Rank 1 inferred the following `['jpeg', 'str']` data format.
Rank 5 inferred the following `['jpeg', 'str']` data format.
Rank 4 inferred the following `['jpeg', 'str']` data format.
Rank 7 inferred the following `['jpeg', 'str']` data format.
Worker 0 is terminating.
Worker 1 is terminating.
Worker 5 is terminating.
Worker 7 is terminating.
Worker 3 is terminating.
Worker 2 is terminating.
Worker 0 is done.
Worker 1 is done.
Worker 5 is done.
Worker 4 is terminating.
Worker 7 is done.
Worker 6 is terminating.
Worker 3 is done.
Worker 2 is done.
Worker 4 is done.
Worker 6 is done.
Workers are finished.
Finished data processing!
